In [1]:
import pandas as pd
import numpy as np
import pyarrow.parquet as pq
import seaborn as sns
import matplotlib.pyplot as plt
from tqdm import tqdm
import os
import time
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import precision_recall_curve, roc_curve, auc

# Trying the 1D-CNN of the smallest sensor dataframe (Heat exchanger)

In [2]:
save_directory = './preprocessed_dataset/VG5/op_0'

control_df = pd.read_parquet(path=f'{save_directory}/control.parquet')
sensors_heat_exchanger = pd.read_parquet(path=f'{save_directory}/sensors_heat_exchanger.parquet')

training_df = pd.concat([control_df,sensors_heat_exchanger], axis=1)

training_df.shape

(99218, 16)

In [3]:
control_df.shape

(99218, 7)

# 1D-CNN for strategy 1 

- X = control signals
- Y = sensors signals

Trying to compute $P(Y|X)$

In [13]:
class TimeSeriesDataset(Dataset):
    def __init__(self, X, Y, sequence_length):
        self.sequence_length = sequence_length
        self.X = torch.FloatTensor(X)
        self.Y = torch.FloatTensor(Y)
        
    def __len__(self):
        return len(self.X) - self.sequence_length + 1
    
    def __getitem__(self, idx):
        x = self.X[idx:idx + self.sequence_length]
        y = self.Y[idx + self.sequence_length - 1]
        return x, y

class CNN1D(nn.Module):
    def __init__(self, input_dim, output_dim, sequence_length):
        super(CNN1D, self).__init__()
        
        self.cnn = nn.Sequential(
            nn.Conv1d(input_dim, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool1d(2),
            
            nn.Conv1d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool1d(2),
            
            nn.Conv1d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(),
        )
        
        # Calculate the size after CNN layers
        cnn_output_size = 128 * (sequence_length // 4)
        
        self.fc = nn.Sequential(
            nn.Linear(cnn_output_size, 128),
            nn.ReLU(),
            nn.Linear(128, output_dim)
        )
        
    def forward(self, x):
        # Input shape: (batch, sequence_length, input_dim)
        # Reshape for CNN: (batch, input_dim, sequence_length)
        x = x.transpose(1, 2)
        x = self.cnn(x)
        x = x.flatten(start_dim=1)
        x = self.fc(x)
        return x

class PyTorchAnomalyDetector:
    def __init__(self, sequence_length=24, batch_size=32, learning_rate=0.001):
        self.sequence_length = sequence_length
        self.batch_size = batch_size
        self.learning_rate = learning_rate
        self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        self.model = None
        self.scaler_X = StandardScaler()
        self.scaler_Y = StandardScaler()
        self.threshold = None
        
    def create_data_loader(self, X, Y, shuffle=True):
        dataset = TimeSeriesDataset(X, Y, self.sequence_length)
        return DataLoader(dataset, batch_size=self.batch_size, shuffle=shuffle)
    
    def fit(self, X, Y, epochs=50):
        # Scale the data
        X_scaled = self.scaler_X.fit_transform(X)
        Y_scaled = self.scaler_Y.fit_transform(Y)
        
        # Create model if not exists
        if self.model is None:
            self.model = CNN1D(
                input_dim=X.shape[1],
                output_dim=Y.shape[1],
                sequence_length=self.sequence_length
            ).to(self.device)
        
        # Create data loader
        train_loader = self.create_data_loader(X_scaled, Y_scaled)
        
        # Define loss and optimizer
        criterion = nn.MSELoss()
        optimizer = optim.Adam(self.model.parameters(), lr=self.learning_rate)
        
        # For tracking progress
        
        print(f"Training on device: {self.device}")
        print(f"Number of batches per epoch: {len(train_loader)}")
        print(f"Input dimension: {X.shape[1]}, Output dimension: {Y.shape[1]}")
        
        # Training loop
        self.model.train()
        for epoch in range(epochs):
            epoch_start_time = time.time()
            total_loss = 0
            
            # Use tqdm for progress bar
            progress_bar = tqdm(train_loader, desc=f'Epoch {epoch+1}/{epochs}')
            
            for batch_idx, (batch_X, batch_Y) in enumerate(progress_bar):
                batch_X = batch_X.to(self.device)
                batch_Y = batch_Y.to(self.device)
                
                optimizer.zero_grad()
                outputs = self.model(batch_X)
                loss = criterion(outputs, batch_Y)
                loss.backward()
                optimizer.step()
                
                total_loss += loss.item()
                
                # Update progress bar
                progress_bar.set_postfix({
                    'loss': f'{loss.item():.4f}',
                    'avg_loss': f'{total_loss/(batch_idx+1):.4f}'
                })
            
            epoch_time = time.time() - epoch_start_time
            avg_loss = total_loss / len(train_loader)
            
            print(f'\nEpoch [{epoch+1}/{epochs}]')
            print(f'Average Loss: {avg_loss:.4f}')
            print(f'Time: {epoch_time:.2f}s')
            print('-' * 50)
            
        # Calculate threshold using training data
        self.model.eval()
        reconstruction_errors = []
        with torch.no_grad():
            for batch_X, batch_Y in train_loader:
                batch_X = batch_X.to(self.device)
                batch_Y = batch_Y.to(self.device)
                outputs = self.model(batch_X)
                errors = torch.mean(torch.square(outputs - batch_Y), dim=1)
                reconstruction_errors.extend(errors.cpu().numpy())
                
        self.threshold = np.percentile(reconstruction_errors, 95)
    
    def detect_anomalies(self, X, Y, return_scores=False):
        X_scaled = self.scaler_X.transform(X)
        Y_scaled = self.scaler_Y.transform(Y)
        
        test_loader = self.create_data_loader(X_scaled, Y_scaled, shuffle=False)
        
        self.model.eval()
        reconstruction_errors = []
        with torch.no_grad():
            for batch_X, batch_Y in test_loader:
                batch_X = batch_X.to(self.device)
                batch_Y = batch_Y.to(self.device)
                outputs = self.model(batch_X)
                errors = torch.mean(torch.square(outputs - batch_Y), dim=1)
                reconstruction_errors.extend(errors.cpu().numpy())
        
        reconstruction_errors = np.array(reconstruction_errors)
        anomalies = reconstruction_errors > self.threshold
        
        if return_scores:
            return anomalies, reconstruction_errors
        return anomalies
    
    def evaluate(self, X_evaluate, Y_evaluate, return_scores=False):
        anomalies = self.detect_anomalies(X_evaluate, Y_evaluate, return_scores=False)

        perc_anomalies = np.sum(anomalies)/len(anomalies)
        if perc_anomalies > 5:
            print("The system is faulty")
        else:
            print("The system looks good")

    def get_synthetic_labels(self, info_df):
        """Extract labels from synthetic test info"""
        if 'fault_start' in info_df.columns and 'fault_end' in info_df.columns:
            labels = np.zeros(len(info_df))
            fault_periods = info_df[['fault_start', 'fault_end']].dropna().values
            
            for start, end in fault_periods:
                labels[start:end] = 1
                
            return labels[self.sequence_length-1:]
        return None
    
    def evaluate_synthetic_with_labels(self, X_test, Y_test, info_df):
        true_labels = self.get_synthetic_labels(info_df)
        if true_labels is None:
            raise ValueError("No fault labels found in info_df")
        
        anomalies, scores = self.detect_anomalies(X_test, Y_test, return_scores=True)
        
        if len(true_labels) > len(anomalies):
            true_labels = true_labels[:len(anomalies)]
        
        fpr, tpr, _ = roc_curve(true_labels, scores)
        precision, recall, _ = precision_recall_curve(true_labels, scores)
        
        return {
            'roc_auc': auc(fpr, tpr),
            'pr_auc': auc(recall, precision),
            'fpr': fpr,
            'tpr': tpr,
            'precision': precision,
            'recall': recall,
            'scores': scores,
            'predictions': anomalies,
            'true_labels': true_labels
        }

    def save_model(self, save_directory, model_name):
        """Save the model to the specified path."""
        os.makedirs(save_directory, exist_ok=True)
        path = save_directory + model_name
        torch.save({
            'model_state_dict': self.model.state_dict(),
            'scaler_X': self.scaler_X,
            'scaler_Y': self.scaler_Y,
            'threshold': self.threshold
        }, path)
        print(f'Model saved to {path}')
    
    def load_model(self, path):
        """Load the model from the specified path."""
        checkpoint = torch.load(path)
        input_dim = self.scaler_X.n_features_in_
        output_dim = self.scaler_Y.n_features_in_
        
        # Reinitialize the model with the saved architecture
        self.model = CNN1D(input_dim=input_dim, output_dim=output_dim, 
                           sequence_length=self.sequence_length).to(self.device)
        self.model.load_state_dict(checkpoint['model_state_dict'])
        self.scaler_X = checkpoint['scaler_X']
        self.scaler_Y = checkpoint['scaler_Y']
        self.threshold = checkpoint['threshold']
        print(f'Model loaded from {path}')

def plot_results(results):
    """Plot evaluation results"""
    import matplotlib.pyplot as plt
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))
    
    # ROC curve
    ax1.plot(results['fpr'], results['tpr'])
    ax1.plot([0, 1], [0, 1], 'k--')
    ax1.set_xlabel('False Positive Rate')
    ax1.set_ylabel('True Positive Rate')
    ax1.set_title(f'ROC Curve (AUC = {results["roc_auc"]:.3f})')
    
    # PR curve
    ax2.plot(results['recall'], results['precision'])
    ax2.set_xlabel('Recall')
    ax2.set_ylabel('Precision')
    ax2.set_title(f'Precision-Recall Curve (AUC = {results["pr_auc"]:.3f})')
    
    plt.tight_layout()
    plt.show()
    
    # Plot anomaly scores
    plt.figure(figsize=(15, 5))
    plt.plot(results['scores'], label='Anomaly Score')
    plt.axhline(y=np.percentile(results['scores'], 95), color='r', linestyle='--', label='Threshold')
    anomaly_points = np.where(results['true_labels'] == 1)[0]
    plt.scatter(anomaly_points, results['scores'][anomaly_points], color='red', label='True Anomalies')
    plt.xlabel('Time')
    plt.ylabel('Anomaly Score')
    plt.title('Anomaly Scores Over Time')
    plt.legend()
    plt.show()

In [18]:
# Initialize detector
detector = PyTorchAnomalyDetector(sequence_length=24, batch_size=32)

# Prepare training data - using only heat exchanger sensors
X_train = control_df  # control variables
Y_train = sensors_heat_exchanger  # heat exchanger sensors only

# Train the model
detector.fit(X_train, Y_train, epochs=10)

Training on device: cpu
Number of batches per epoch: 3100
Input dimension: 7, Output dimension: 9


Epoch 1/10: 100%|██████████| 3100/3100 [00:30<00:00, 102.94it/s, loss=0.6845, avg_loss=0.4838]



Epoch [1/10]
Average Loss: 0.4838
Time: 30.12s
--------------------------------------------------


Epoch 2/10: 100%|██████████| 3100/3100 [00:30<00:00, 101.10it/s, loss=0.2769, avg_loss=0.3257]



Epoch [2/10]
Average Loss: 0.3257
Time: 30.67s
--------------------------------------------------


...


Epoch [8/10]
Average Loss: 0.1531
Time: 29.35s
--------------------------------------------------


Epoch 9/10: 100%|██████████| 3100/3100 [00:29<00:00, 104.11it/s, loss=0.1479, avg_loss=0.1442]



Epoch [9/10]
Average Loss: 0.1442
Time: 29.78s
--------------------------------------------------


Epoch 10/10: 100%|██████████| 3100/3100 [00:29<00:00, 104.46it/s, loss=0.1045, avg_loss=0.1377]



Epoch [10/10]
Average Loss: 0.1377
Time: 29.68s
--------------------------------------------------


In [19]:
save_directory = "models/VG5/strategy_1/"
model_name = "1D_CNN_Heat_Exchanger.pt"

detector.save_model(save_directory, model_name)

Model saved to models/VG5/strategy_1/1D_CNN_Heat_Exchanger.pt


In [20]:
anomalies, reconstruction_errors =detector.detect_anomalies(X_train, Y_train, return_scores=True)

In [21]:
np.sum(anomalies)/len(anomalies)*100

5.000252028832099

In [22]:
np.max(reconstruction_errors)

9.184759

In [23]:
save_directory = './preprocessed_dataset/VG5/s_01'

test_X = pd.read_parquet(path=f'{save_directory}/control.parquet')
test_Y = pd.read_parquet(path=f'{save_directory}/sensors_heat_exchanger.parquet')

In [24]:
X_train.shape

(99218, 7)

In [25]:
test_X.shape

(45229, 7)

In [26]:
# Evaluate on synthetic test data
anomalies_syn, reconstruction_errors_syn = detector.detect_anomalies(test_X, test_Y, 
                                                                     return_scores= True)

In [27]:
np.sum(anomalies_syn)/len(anomalies_syn)*100

99.56200504357828

In [28]:
### Evaluating on the second operation period

save_directory = './preprocessed_dataset/VG5/op_1'

valid_X = pd.read_parquet(path=f'{save_directory}/control.parquet')
valid_Y = pd.read_parquet(path=f'{save_directory}/sensors_heat_exchanger.parquet')

valid_X.shape

(57864, 7)

In [29]:
# Evaluate on synthetic test data
anomalies_valid, reconstruction_errors_valid = detector.detect_anomalies(valid_X, valid_Y, 
                                                                     return_scores= True)

In [ ]:
np.sum(anomalies_valid)/len(anomalies_valid)*100

98.1449145070106

## 1D-CNN for strategy 2

TODO list

- Create the Dataloader (already done)
- Create the 1D-CNN neural networks
- Create the optimizer
- Train the model
- Check if it is working

In [58]:
class TimeSeriesDataset(Dataset):
    def __init__(self, X, Y, sequence_length):
        self.sequence_length = sequence_length
        self.X = torch.FloatTensor(X)
        self.Y = torch.FloatTensor(Y)
        
    def __len__(self):
        return len(self.X) - self.sequence_length + 1
    
    def __getitem__(self, idx):
        x = self.X[idx:idx + self.sequence_length]
        y = self.Y[idx + self.sequence_length - 1]
        return x, y

def init_weights(m):
    if isinstance(m, nn.BatchNorm1d):
        m.weight.data.fill_(1.0)
        m.bias.data.zero_()
    elif isinstance(m, nn.Conv1d) or isinstance(m, nn.Linear):
        m.weight.data = nn.init.xavier_uniform_(
            m.weight.data, gain=nn.init.calculate_gain('relu'))
        if m.bias is not None:
            m.bias.data.zero_()

class CNN1D(nn.Module):
        
    """
    Args:
        n_features (int, optional): number of input features. Defaults to 18.
        sequence_length (int, optional): sequence length. Defaults to 50.
        n_ch (int, optional): number of channels (filter size). Defaults to 10.
        n_k (int, optional): kernel size. Defaults to 10.
        n_hidden (int, optional): number of hidden neurons for regressor. Defaults to 50.
        n_layers (int, optional): number of convolution layers. Defaults to 5.
    """
    
    def __init__(self, 
                 in_channels=18, 
                 out_channels=1,
                 sequence_length=50, 
                 n_ch=20, 
                 n_k=10, 
                 n_hidden=50, 
                 n_layers=3,
                 dropout=0.0,
                 padding='same',
                 use_batchnorm=False):
        super().__init__()
        
        # Create a ModuleList to hold variable number of conv layers
        self.features_extractor = nn.ModuleList()
        
        # First layer (input layer)
        self.features_extractor.append(nn.Sequential(
            nn.Conv1d(in_channels, n_ch, kernel_size=n_k, padding=padding),
            nn.BatchNorm1d(n_ch) if use_batchnorm else nn.Identity(),
            nn.ReLU(),
            nn.Dropout(dropout)
        ))
        
        for _ in range(n_layers - 2):  
            self.features_extractor.append(nn.Sequential(
                nn.Conv1d(n_ch, n_ch, kernel_size=n_k, padding=padding),
                nn.BatchNorm1d(n_ch) if use_batchnorm else nn.Identity(),
                nn.ReLU(),
                nn.Dropout(dropout)
            ))

        self.features_extractor.append(nn.Sequential(
            nn.Conv1d(n_ch, 1, kernel_size=n_k, padding=padding),
            nn.BatchNorm1d(1) if use_batchnorm else nn.Identity(),
            nn.ReLU(),
            nn.Dropout(dropout)
        ))
        
        flat_features = sequence_length  

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(flat_features, n_hidden),
            nn.BatchNorm1d(n_hidden) if use_batchnorm else nn.Identity(),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(n_hidden, out_channels),  # Output layer
            nn.Sigmoid() # Output logistic regression
        )
        
        # Initialize weights
        self.apply(init_weights)
        
    def forward(self, x):
        # Pass through all conv layers sequentially
        for layer in self.features_extractor:
            x = layer(x)
        x = self.classifier(x)
        return x
    

# Input:          [256, 18, 50]  # (batch, features, sequence_length)
# After layer1:   [256, 10, 50]  # (batch, channels, sequence_length)
# After layer2:   [256, 10, 50]  # (batch, channels, sequence_length)
# After layer3:   [256, 1, 50]   # (batch, channels, sequence_length)
# After flatten:  [256, 50]      # (batch, flattened)
# Final output:   [256, 1]       # (batch, prediction)

class PyTorchAnomalyDetector:
    def __init__(self, sequence_length=50, batch_size=256, learning_rate=0.001):
        self.sequence_length = sequence_length
        self.batch_size = batch_size
        self.learning_rate = learning_rate
        self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        self.model = None
        self.scaler_X = StandardScaler()
        self.scaler_Y = StandardScaler()
        self.threshold = None
        
    def create_data_loader(self, X, Y):
        dataset = TimeSeriesDataset(X, Y, self.sequence_length)
        return DataLoader(dataset, batch_size=self.batch_size, shuffle=False)
    
    def fit(self, X, Y, epochs=50):
        # Scale the data
        X_scaled = self.scaler_X.fit_transform(X)
        Y_scaled = self.scaler_Y.fit_transform(Y.reshape(-1, 1)).flatten()
        
        # Get the number of features (input channels) from X
        in_channels = X_scaled.shape[1]  # Number of features in input data
        
        # Create model if not exists
        if self.model is None:
            self.model = CNN1D(
                in_channels=in_channels, 
                out_channels=1,
                sequence_length=self.sequence_length, 
                n_ch=20, 
                n_k=10, 
                n_hidden=50, 
                n_layers=3,
                dropout=0.0,
                padding='same',
                use_batchnorm=False
            ).to(self.device)
        
        # Create data loader
        train_loader = self.create_data_loader(X_scaled, Y_scaled)
        
        # Define loss and optimizer
        criterion = nn.BCELoss()
        optimizer = optim.Adam(self.model.parameters(), lr=self.learning_rate)
        
        # For tracking progress
        
        print(f"Training on device: {self.device}")
        print(f"Number of batches per epoch: {len(train_loader)}")
        print(f"Input dimension: {X.shape[1]}, Output dimension: {Y.shape}")
        
        # Training loop
        self.model.train()
        for epoch in range(epochs):
            epoch_start_time = time.time()
            total_loss = 0
            
            # Use tqdm for progress bar
            progress_bar = tqdm(train_loader, desc=f'Epoch {epoch+1}/{epochs}')
            
            for batch_idx, (batch_X, batch_Y) in enumerate(progress_bar):
                batch_X = batch_X.transpose(1,2).to(self.device)
                batch_Y = batch_Y.to(self.device)
                
                optimizer.zero_grad()
                outputs = self.model(batch_X)
                loss = criterion(outputs, batch_Y.unsqueeze(-1))
                loss.backward()
                optimizer.step()
                
                total_loss += loss.item()
                
                # Update progress bar
                progress_bar.set_postfix({
                    'loss': f'{loss.item():.4f}',
                    'avg_loss': f'{total_loss/(batch_idx+1):.4f}'
                })
            
            epoch_time = time.time() - epoch_start_time
            avg_loss = total_loss / len(train_loader)
            
            print(f'\nEpoch [{epoch+1}/{epochs}]')
            print(f'Average Loss: {avg_loss:.4f}')
            print(f'Time: {epoch_time:.2f}s')
            print('-' * 50)
            
        # Calculate threshold using training data
        self.model.eval()
        reconstruction_errors = []
        with torch.no_grad():
            for batch_X, batch_Y in train_loader:
                batch_X = batch_X.transpose(1,2).to(self.device)
                batch_Y = batch_Y.to(self.device)
                outputs = self.model(batch_X)
                errors = torch.mean(torch.square(outputs - batch_Y), dim=1)
                reconstruction_errors.extend(errors.cpu().numpy())
                
        self.threshold = np.percentile(reconstruction_errors, 95)
    
    def detect_anomalies(self, X, Y, return_scores=False):
        X_scaled = self.scaler_X.transform(X)
                
        test_loader = self.create_data_loader(X_scaled, self.scaler_Y, shuffle=False)
        
        self.model.eval()
        reconstruction_errors = []
        with torch.no_grad():
            for batch_X, batch_Y in test_loader:
                batch_X = batch_X.transpose(1,2).to(self.device)
                batch_Y = batch_Y.to(self.device)
                outputs = self.model(batch_X)
                errors = torch.mean(torch.square(outputs - batch_Y), dim=1)
                reconstruction_errors.extend(errors.cpu().numpy())
        
        reconstruction_errors = np.array(reconstruction_errors)
        anomalies = reconstruction_errors > self.threshold
        
        if return_scores:
            return anomalies, reconstruction_errors
        return anomalies
    
    def evaluate(self, X_evaluate, Y_evaluate, return_scores=False):
        anomalies = self.detect_anomalies(X_evaluate, Y_evaluate, return_scores=False)

        perc_anomalies = np.sum(anomalies)/len(anomalies)
        if perc_anomalies > 5:
            print("The system is faulty")
        else:
            print("The system looks good")

    def get_synthetic_labels(self, info_df):
        """Extract labels from synthetic test info"""
        if 'fault_start' in info_df.columns and 'fault_end' in info_df.columns:
            labels = np.zeros(len(info_df))
            fault_periods = info_df[['fault_start', 'fault_end']].dropna().values
            
            for start, end in fault_periods:
                labels[start:end] = 1
                
            return labels[self.sequence_length-1:]
        return None
    
    def evaluate_synthetic_with_labels(self, X_test, Y_test, info_df):
        true_labels = self.get_synthetic_labels(info_df)
        if true_labels is None:
            raise ValueError("No fault labels found in info_df")
        
        anomalies, scores = self.detect_anomalies(X_test, Y_test, return_scores=True)
        
        if len(true_labels) > len(anomalies):
            true_labels = true_labels[:len(anomalies)]
        
        fpr, tpr, _ = roc_curve(true_labels, scores)
        precision, recall, _ = precision_recall_curve(true_labels, scores)
        
        return {
            'roc_auc': auc(fpr, tpr),
            'pr_auc': auc(recall, precision),
            'fpr': fpr,
            'tpr': tpr,
            'precision': precision,
            'recall': recall,
            'scores': scores,
            'predictions': anomalies,
            'true_labels': true_labels
        }

    def save_model(self, save_directory, model_name):
        """Save the model to the specified path."""
        os.makedirs(save_directory, exist_ok=True)
        path = save_directory + model_name
        torch.save({
            'model_state_dict': self.model.state_dict(),
            'scaler_X': self.scaler_X,
            'scaler_Y': self.scaler_Y,
            'threshold': self.threshold
        }, path)
        print(f'Model saved to {path}')
    
    def load_model(self, path):
        """Load the model from the specified path."""
        checkpoint = torch.load(path)
        input_dim = self.scaler_X.n_features_in_
        output_dim = self.scaler_Y.n_features_in_
        
        # Reinitialize the model with the saved architecture
        self.model = CNN1D(input_dim=input_dim, output_dim=output_dim, 
                           sequence_length=self.sequence_length).to(self.device)
        self.model.load_state_dict(checkpoint['model_state_dict'])
        self.scaler_X = checkpoint['scaler_X']
        self.scaler_Y = checkpoint['scaler_Y']
        self.threshold = checkpoint['threshold']
        print(f'Model loaded from {path}')

def plot_results(results):
    """Plot evaluation results"""
    import matplotlib.pyplot as plt
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))
    
    # ROC curve
    ax1.plot(results['fpr'], results['tpr'])
    ax1.plot([0, 1], [0, 1], 'k--')
    ax1.set_xlabel('False Positive Rate')
    ax1.set_ylabel('True Positive Rate')
    ax1.set_title(f'ROC Curve (AUC = {results["roc_auc"]:.3f})')
    
    # PR curve
    ax2.plot(results['recall'], results['precision'])
    ax2.set_xlabel('Recall')
    ax2.set_ylabel('Precision')
    ax2.set_title(f'Precision-Recall Curve (AUC = {results["pr_auc"]:.3f})')
    
    plt.tight_layout()
    plt.show()
    
    # Plot anomaly scores
    plt.figure(figsize=(15, 5))
    plt.plot(results['scores'], label='Anomaly Score')
    plt.axhline(y=np.percentile(results['scores'], 95), color='r', linestyle='--', label='Threshold')
    anomaly_points = np.where(results['true_labels'] == 1)[0]
    plt.scatter(anomaly_points, results['scores'][anomaly_points], color='red', label='True Anomalies')
    plt.xlabel('Time')
    plt.ylabel('Anomaly Score')
    plt.title('Anomaly Scores Over Time')
    plt.legend()
    plt.show()

In [59]:
# Initialize detector
detector = PyTorchAnomalyDetector(sequence_length=50, batch_size=256)

# Prepare training data - using only heat exchanger sensors
X_train = pd.concat([control_df,sensors_heat_exchanger],axis=1)  # control variables
Y_train = np.array([1 for i in range(X_train.shape[0])])  # heat exchanger sensors only

# Train the model
detector.fit(X_train, Y_train, epochs=1)

Training on device: cpu
Number of batches per epoch: 388
Input dimension: 16, Output dimension: (99218,)


Epoch 1/1: 100%|██████████| 388/388 [00:17<00:00, 22.17it/s, loss=0.0000, avg_loss=0.0085]



Epoch [1/1]
Average Loss: 0.0085
Time: 17.51s
--------------------------------------------------


### Snippets of code

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F


class FeatureExtractor(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv1d(1, 10, 3, padding=1)
        self.bn1 = nn.BatchNorm1d(10)  ###Add batch norm
        self.conv2 = nn.Conv1d(10, 10, 3, padding=1)
        self.bn2 = nn.BatchNorm1d(10)  ###Add batch norm
        self.conv3 = nn.Conv1d(10, 10, 3, padding=1)
        self.bn3 = nn.BatchNorm1d(10)  ###Add batch norm
        self.dropout = nn.Dropout(0.1)
        self.fc = nn.Linear(10 * 512, 256)  # 512 is input dimension

    def forward(self, x):
        x = self.dropout(F.relu(self.bn1(self.conv1(x))))
        x = self.dropout(F.relu(self.bn2(self.conv2(x))))
        x = self.dropout(F.relu(self.bn3(self.conv3(x))))
        x = x.view(x.size(0), -1)
        x = F.relu(self.fc(x))
        return x


class Classifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(256, 256)
        self.bn1 = nn.BatchNorm1d(256)
        self.fc2 = nn.Linear(256, 10)

    def forward(self, x):
        x = F.relu(self.bn1(self.fc1(x)))
        x = self.fc2(x)
        return F.log_softmax(x, dim=1)


class BaselineModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.feature_extractor = FeatureExtractor()
        self.classifier = Classifier()

    def forward(self, x):
        features = self.feature_extractor(x)
        return self.classifier(features)

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch import linalg as LA
import torch.optim as optim
import numpy as np
from tqdm import tqdm

def train_baseline(model, source_loader, target_loader, args, device):
    """Standard source training"""
    print("\nTraining Baseline Model...")

    # Included this to save the metric
    with open("results/baseline_metrics.txt", "w") as f:
        f.write("")

    optimizer = optim.Adam(model.parameters(), lr=args.lr)

    for epoch in range(args.epochs):
        model.train()
        total_loss = 0

        for data, target in tqdm(source_loader, desc=f"Epoch {epoch}"):
            data, target = data.to(device), target.to(device)

            optimizer.zero_grad()
            output = model(data)
            loss = F.nll_loss(output, target)
            loss.backward()
            optimizer.step()

            total_loss += loss.item()

        # Calculate training and testing metrics
        source_loss, source_acc = evaluate(model, source_loader, device)
        target_loss, target_acc = evaluate(model, target_loader, device)

        # Save metrics for plotting
        save_metrics("baseline", epoch, source_loss, source_acc, target_acc)

        # Print loss and accuracy for source and target
        print(f"Training loss : {source_loss} - accuracy : {source_acc}")
        print(f"Test loss : {target_loss} - accuracy : {target_acc}")

    # Save final model
    torch.save(model.state_dict(), "final_baseline.pth")

    # Return Final target accuracy
    return target_acc


def evaluate(model, loader, device):
    model.eval()
    total_loss = 0
    correct = 0
    total = 0

    with torch.no_grad():
        for data, target in loader:
            data, target = data.to(device), target.to(device)
            output = model(data)
            total_loss += F.nll_loss(output, target).item()
            pred = output.max(1)[1]
            correct += pred.eq(target).sum().item()
            total += target.size(0)

    return total_loss / len(loader), correct / total


# Save metric to txt file for backup
def save_metrics(method_name, epoch, source_loss, source_acc, target_acc):
    with open(f"results/{method_name}_metrics.txt", "a") as f:
        f.write(f"{epoch},{source_loss},{source_acc},{target_acc}\n")

### Example of a 1D-CNN by chatgpt

Output Layer:

Use a Dense layer with 1 neuron and a sigmoid activation function for binary classification.

In [ ]:
# Define the input shape
input_shape = (100000, 16)

# Build the model
model = Sequential()

# Add convolutional layers
model.add(Conv1D(filters=32, kernel_size=3, activation='relu', input_shape=input_shape))
model.add(BatchNormalization())
model.add(MaxPooling1D(pool_size=2))

model.add(Conv1D(filters=64, kernel_size=3, activation='relu'))
model.add(BatchNormalization())
model.add(MaxPooling1D(pool_size=2))

model.add(Conv1D(filters=128, kernel_size=3, activation='relu'))
model.add(BatchNormalization())
model.add(MaxPooling1D(pool_size=2))

# Flatten the output from the convolutional layers
model.add(Flatten())

# Add dense layers to learn higher-level features
model.add(Dense(128, activation='relu'))
model.add(Dropout(0.5))  # Dropout layer to prevent overfitting

model.add(Dense(64, activation='relu'))
model.add(Dropout(0.5))

# Output layer for binary classification
model.add(Dense(1, activation='sigmoid'))

# Compile the model
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

# Print the model summary
model.summary()

In [4]:
import numpy as np
from sklearn.utils import shuffle

# Example data
X = np.array([[1, 2], [3, 4], [5, 6], [7, 8], [9, 10]])
y = np.array([0, 1, 0, 1, 0])

# Shuffle the data
X_shuffled, y_shuffled = shuffle(X, y, random_state=42)

print("Original X:")
print(X)
print("Shuffled X:")
print(X_shuffled)
print("Original y:")
print(y)
print("Shuffled y:")
print(y_shuffled)

Original X:
[[ 1  2]
 [ 3  4]
 [ 5  6]
 [ 7  8]
 [ 9 10]]
Shuffled X:
[[ 3  4]
 [ 9 10]
 [ 5  6]
 [ 1  2]
 [ 7  8]]
Original y:
[0 1 0 1 0]
Shuffled y:
[1 0 0 0 1]
